# Supplementary Code to Report

Here we list all the dependencies to run each code chunk:

In [ ]:
%pip install csv json pathlib ultralytics rfdetr cv2 numpy matplotlib.pyplot sklearn.metrics torch pandas

## Section 1 - Dog Detection and Cropping

### Section 1.1 - Training and Validation Data Extraction

DO NOT RUN ANY SCRIPT! Notice - training data downloads 5000 training images and 1000 validation images locally. Validation images can be found on GitHub, training cannot as the folder size is too large. 

### Section 1.2 - Model Training

DO NOT RUN ANY MODELS - on RTX 2050, finetuning YOLO26n took 3 hours and RF-DETR-Nano took 11 hours, all models are saved as `.pth` or `.pt` files and can be run safely on a computer, demonstrated in Section 1.3. YOLO26n files initialise very quickly, RF-DETR-Nano files take slightly longer to warm up but once the model is loaded it runs fine.  

### Section 1.3 - Validation

All chunks can be run, validation images are stored on GitHub. 

In [ ]:
import csv
import json
from pathlib import Path
from ultralytics import YOLO
from rfdetr import RFDETRNano
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


#### 1.3.1 Configuration

Here we initialise the family of model (YOLO or RF-DETR), the type of model (basline or finetuned), and then the filepath based on which `.pth` or `.pt` file we are running for our validation. Options are listed in comment below, varying the inputs will allow access to different validations. We set a minimum threhold confidence to allow most predictions to pass, as well as a set IoU (Intersection over Union) threshold for box similarity, and over the course of our validation we will loop over image confidence thresholds in $[0.05, 0.95]$ across $0.05$ gaps. Image size is standardised for both models at `640px` since both models were trained on this fixed size.  

In [ ]:
MODEL_FAMILY = "YOLO"  # "YOLO" or "RFDETR" (output directory prefix)
MODEL_TYPE = "finetuned"  # "baseline" or "finetuned"
MODEL_PATH = None  # None selects default pretrained weights for baseline only.
# YOLO baseline:  MODEL_FAMILY="YOLO"; MODEL_TYPE="baseline"; MODEL_PATH="yolo26n.pt"
# YOLO finetuned: MODEL_FAMILY="YOLO"; MODEL_TYPE="finetuned";
MODEL_PATH="runs/detect/yolo26n_dog-3/weights/best.pt"
# RFDETR baseline: MODEL_FAMILY="RFDETR"; MODEL_TYPE="baseline"; MODEL_PATH=None
# RFDETR finetuned: MODEL_FAMILY="RFDETR"; MODEL_TYPE="finetuned";
#MODEL_PATH="runs/rfdetr_nano_dog_only/checkpoint_best_total.pth"
# A compatible last.ckpt may also be supplied on RF-DETR versions supporting it.
DOG_CLASS_ID_OVERRIDE = None  # Set only if the checkpoint's prediction ID is verified.
VAL_IMAGE_DIR = Path("DogCOCO/images/val")
VAL_LABEL_DIR = Path("DogCOCO/labels/val")
MIN_PRED_CONF = 0.001
IOU_MATCH_THRESHOLD = 0.50
THRESHOLDS = np.arange(0.05, 1.00, 0.05)  # exact original grid
MISSING_LABELS_ARE_NEGATIVE = False
YOLO_IMAGE_SIZE = 640
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

#### 1.3.2 - Directory and CSV Management

Here we define 4 file manipulation functions, in order to read and write our results to CSVs from which to create plots. `save_csv` inputs a collection of dictionaries and writes them to CSVs; `save_image` saves the cropped image whose coordinates are outputted by the trained models; `get_image_paths` finds where the validation images are located and allows them to be inputted to model, and `prepare_output_directory` creates the file hierarchy for each model when performing validation, written as: 
```
alt/
└── YOLO_results/
    └── validation_analysis/
        └── finetuned/
            ├── dog_crops/
            └── box_diagnostics/
```

In [ ]:
def save_csv(path, rows, fieldnames=None):
    """Retain the original CSV field order, including empty diagnostic tables."""
    if fieldnames is None:
        fieldnames = list(rows[0].keys())
    with path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_image(path, image):
    if not cv2.imwrite(str(path), image):
        raise IOError(f"Could not save {path}")


def get_image_paths(image_dir):
    paths = sorted(p for p in image_dir.iterdir()
                   if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not paths:
        raise ValueError(f"No validation images found in {image_dir}")
    return paths


def prepare_output_directory(model_family, model_type):
    output_dir = Path("alt") / f"{model_family}_results" / "validation_analysis" / model_type
    # Avoid mixing old crops/diagnostics (or old semantic results) with a new run.
    if output_dir.exists() and any(output_dir.iterdir()):
        raise FileExistsError(f"Output directory is not empty: {output_dir}. "
                              "Move or clear it before rerunning; no files have been deleted.")
    for name in ("dog_crops", "box_diagnostics"):
        (output_dir / name).mkdir(parents=True, exist_ok=True)
    return output_dir

#### 1.3.3 - Model Loading Functions

We load the corresponding model depending on our validation input in `load_model` depending on if we chose RF-DETR-Nano or YOLO and subsequently the specific model to run. `normalise_class_names` exists to correctly identify the class names in the COCO dataset that both models were trained with, but use different naming syntax and use different numbers for different objects itn classifies. In our finetuned model, the class number for "dogs" is "0" because they focus on only one class, whereas YOLO uses "16" and RF-DETR-Nano uses "18", which is obtained from `find_dog_class_id` and `get_dog_class_id`. 

In [ ]:
def load_model(model_family, model_type, model_path=None):
    if model_family not in ("YOLO", "RFDETR"):
        raise ValueError("MODEL_FAMILY must be YOLO or RFDETR")
    if model_type not in ("baseline", "finetuned"):
        raise ValueError("MODEL_TYPE must be baseline or finetuned")
    if model_type == "finetuned" and not model_path:
        raise ValueError("A fine-tuned checkpoint path is required")
    if model_family == "YOLO":
        from ultralytics import YOLO
        return YOLO(str(model_path) if model_path else "yolo26n.pt")
    from rfdetr import RFDETRNano
    if model_type == "baseline":
        return RFDETRNano(pretrain_weights=str(model_path)) if model_path else RFDETRNano()
    return RFDETRNano.from_checkpoint(str(model_path))


def normalise_class_names(names):
    items = names.items() if isinstance(names, dict) else enumerate(names)
    return {int(i): str(name).strip().lower() for i, name in items}


def find_dog_class_id(names):
    matches = [i for i, name in normalise_class_names(names).items() if name == "dog"]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one dog class in model metadata; got {matches}")
    return matches[0]


def get_dog_class_id(model, model_family, model_type, override=None):
    """None means use the fine-tuned RF-DETR detection-name metadata."""
    if override is not None:
        if int(override) != override or override < 0:
            raise ValueError("DOG_CLASS_ID_OVERRIDE must be a non-negative integer")
        return int(override)
    if model_family == "YOLO":
        return find_dog_class_id(model.names)
    if model_family == "RFDETR" and model_type == "baseline":
        from rfdetr.assets.coco_classes import COCO_CLASSES
        return find_dog_class_id(COCO_CLASSES)  # sparse COCO ID 18, not list index 16
    if model_family == "RFDETR" and model_type == "finetuned":
        return None  # use result.data["class_name"]; never use pretrained COCO IDs
    raise ValueError(f"Unsupported model: {model_family}/{model_type}")

#### 1.3.4 - Prediction
In order for every metric to be able to take the same input regardless of model type, we need a form of standardisation of prediction outputs that ensures regardless of model the prediction output will be of a specific type. We standardise it to be the box coordinates $b_j=[x_1, x_2, y_1, y_2]$ and the confidence $c_j \in [0, 1]$ such that every prediction is $P_i = \{(b_i, c_i): \text{class}_i = \text{dog}\}$, created in `standardise_predictions`. Then, for a given image, we predict the output of either the YOLO or RF-DETR-Nano model and save the box coordinates and confidence of each dog detected in an image, of which there can be multiple, and `detect_dogs` in this file runs the model family checker. 

In [ ]:
def standardise_predictions(boxes, scores, accepted):
    """Return original-image xyxy boxes and confidences, in detector output order."""
    if not (len(boxes) == len(scores) == len(accepted)):
        raise ValueError("Detector returned inconsistent prediction lengths")
    return [{"box": box.tolist(), "confidence": float(score)}
            for box, score, keep in zip(np.asarray(boxes), scores, accepted) if keep]


def predict_yolo(model, image_path, dog_class_id, confidence):
    result = model.predict(source=str(image_path), classes=[dog_class_id],
                           conf=confidence, imgsz=YOLO_IMAGE_SIZE, verbose=False)[0]
    if result.boxes is None or len(result.boxes) == 0:
        return []
    boxes = result.boxes.xyxy.cpu().numpy()
    scores = result.boxes.conf.cpu().numpy()
    class_ids = result.boxes.cls.cpu().numpy()
    return standardise_predictions(boxes, scores, class_ids == dog_class_id)


def predict_rfdetr(model, image_path, dog_class_id, confidence):
    result = model.predict(images=str(image_path), threshold=confidence)
    if result is None or len(result) == 0:
        return []
    class_ids = np.asarray(result.class_id)
    if dog_class_id is not None:
        # Baseline: official sparse ID. Custom: explicitly verified override.
        accepted = class_ids == dog_class_id
    else:
        # Current RF-DETR versions resolve custom IDs using checkpoint metadata.
        names = getattr(result, "data", {}).get("class_name")
        if names is None:
            # Old versions may not provide class-name metadata. Prefer a real ID
            # mapping, and do not guess a custom ID from a list's position.
            mapping = getattr(model, "class_names", None)
            if not isinstance(mapping, dict):
                raise RuntimeError("RF-DETR custom prediction class names unavailable. "
                                   "Verify the checkpoint's dog prediction ID and set "
                                   "DOG_CLASS_ID_OVERRIDE (often 0 on current versions).")
            accepted = class_ids == find_dog_class_id(mapping)
        else:
            accepted = [str(name).strip().lower() == "dog" for name in names]
    return standardise_predictions(result.xyxy, result.confidence, accepted)


def detect_dogs(model, image_path, model_family, dog_class_id, confidence=MIN_PRED_CONF):
    if model_family == "YOLO":
        return predict_yolo(model, image_path, dog_class_id, confidence)
    if model_family == "RFDETR":
        return predict_rfdetr(model, image_path, dog_class_id, confidence)
    raise ValueError(f"Unsupported model family: {model_family}")

#### 1.3.5 - Actual vs Predicted Box Matching

In order to compare our predicted box outputs to the true values, we load the "ground truth" labels to determine a) whether we correctly classified the number of dogs in an image; and b) whether the boxes we predicted overlap with at least 50% of the true box. In the COCO dataset, the coordinates are different to the ones we have standardised, where instead $W$ represents the width of the image, $H$ is the image height, $x_c, y_c$ are the normalised centre coordinates of the box, and $b_w, b_h$ are the normalised box dimensions. We must then transform them to match our version: $$x_1 = \bigg(x_c-\frac{b_w}{2}\bigg)W,$$ $$y_1 = \bigg(y_c-\frac{b_h}{2}\bigg)H,$$ $$x_2 = \bigg(x_c+\frac{b_w}{2}\bigg)W,$$ $$y_2 = \bigg(y_c+\frac{b_h}{2}\bigg)H.$$ In order to prevent the box going outside of the image, we clip the $x$ and $y$ values to become: $$x \leftarrow \max(0, \min(W, x)),$$ $$y \leftarrow \max(0, \min(H, y)).$$ This is done in `load_ground_truth`. 

In order to calculate the proportion of the ground truth image the prediction covers, with respect to the size of the prediction box itself, we use the metric: $$\text{IoU}(A, B) = \frac{|A\cap B|}{|A \cup B} \in (0.5, 1].$$ This is calculated in `calculate_IoU`. We then need a way of seeing which predicted box matches the ground truth boxes the best, since there may be multiple dogs in a photo, we must verify that the image we have predicted is indeed the dog and not something random. This is established by `match_boxes` which compares the prediction across all labelled box and assigns it to the one it is closest to within the threshold of 0.5. 

In [ ]:
def load_ground_truth(label_path, image_width, image_height):
    if not label_path.exists():
        if MISSING_LABELS_ARE_NEGATIVE:
            return []
        raise FileNotFoundError(
            f"Missing validation label: {label_path}"
        )
    boxes = []
    lines = label_path.read_text().splitlines()
    for line in lines:
        if not line.strip():
            continue
        values = line.split()
        if len(values) != 5:
            raise ValueError(
                f"Invalid YOLO label in {label_path}: {line}"
            )
        class_id, xc, yc, bw, bh = map(float, values)
        # Our dataset should contain only class 0: dog
        if int(class_id) != 0:
            raise ValueError(
                f"Unexpected class {class_id} in {label_path}"
            )
        # Convert normalised centre coordinates
        # into original image pixel coordinates.
        x1 = (xc - bw / 2) * image_width
        y1 = (yc - bh / 2) * image_height
        x2 = (xc + bw / 2) * image_width
        y2 = (yc + bh / 2) * image_height
        # Clip coordinates to image boundaries
        x1 = max(0, min(image_width, x1))
        x2 = max(0, min(image_width, x2))
        y1 = max(0, min(image_height, y1))
        y2 = max(0, min(image_height, y2))
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
    return boxes


def calculate_iou(box_a, box_b):
    # Intersection coordinates
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])
    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)
    intersection_area = (
        intersection_width * intersection_height
    )
    # Individual box areas
    area_a = (
        (box_a[2] - box_a[0]) *
        (box_a[3] - box_a[1])
    )
    area_b = (
        (box_b[2] - box_b[0]) *
        (box_b[3] - box_b[1])
    )
    union_area = area_a + area_b - intersection_area
    if union_area <= 0:
        return 0.0
    return intersection_area / union_area


def match_boxes(predictions, ground_truth, iou_threshold=0.5):
    # Construct possible prediction / ground-truth matches
    pairs = []
    for pred_index, pred in enumerate(predictions):
        for gt_index, gt in enumerate(ground_truth):
            iou = calculate_iou(
                pred["box"],
                gt
            )
            pairs.append(
                (iou, pred_index, gt_index)
            )
    # Highest IoU matches first
    pairs.sort(reverse=True)
    matched_predictions = set()
    matched_ground_truth = set()
    # Each predicted box and ground-truth box
    # can only be matched once.
    for iou, pred_index, gt_index in pairs:
        if iou < iou_threshold:
            break
        if pred_index in matched_predictions:
            continue
        if gt_index in matched_ground_truth:
            continue
        matched_predictions.add(pred_index)
        matched_ground_truth.add(gt_index)
    return matched_predictions, matched_ground_truth

#### Section 1.3.6 - Prediction Saving

In order to avoid repetition, we save all our results first before calculating across all thresholds, as the confidence from our model doesn't change depending on our subjective threshold, rather the amalgamated confidences of our model is what provides the inference for its specific confidence. We save the path of the image, the ground truth boxes, whether a dog is truly present, the predictions per dog, and the maximum confidence of a dog our model has outputted for a single image. This can remove images below the threshold immediately rather than first checking each dog is below the threshold, and if the maximum confidence passes then we can look at each individual dog, which is saved in `collect_validation_records`. In `save_prediction_records`, we save it as a JSON file saves them for re-use without having to run all validation predictions again, which can be loaded by `load_prediction_records`.  

In [ ]:
def collect_validation_records(model, image_paths, label_dir, model_family, dog_class_id):
    records = []
    for i, image_path in enumerate(image_paths, 1):
        image = cv2.imread(str(image_path))
        if image is None:
            # Preserve reference behaviour: skip unreadable images with a message.
            print("Could not open:", image_path)
            continue
        height, width = image.shape[:2]
        gt_boxes = load_ground_truth(label_dir / f"{image_path.stem}.txt", width, height)
        predictions = detect_dogs(model, image_path, model_family, dog_class_id)
        max_confidence = max((p["confidence"] for p in predictions), default=0.0)
        records.append({"image_path": image_path, "gt_boxes": gt_boxes,
                        "actual_dog": bool(gt_boxes), "predictions": predictions,
                        "max_confidence": max_confidence})
        print(f"{i}/{len(image_paths)} {image_path.name} | "
              f"Actual dog: {bool(gt_boxes)} | Maximum confidence: {max_confidence:.3f}")
    if not records:
        raise ValueError("No readable validation images")
    return records


def save_prediction_records(records, output_dir):
    """Additional requested output: all returned dog boxes/scores before sweep."""
    serialisable = [dict(r, image_path=str(r["image_path"])) for r in records]
    with (output_dir / "prediction_records.json").open("w", encoding="utf-8") as f:
        json.dump(serialisable, f, indent=2, allow_nan=False)


def load_prediction_records(path):
    """Reload the inference cache for metrics/plots without loading a model."""
    with Path(path).open(encoding="utf-8") as f:
        records = json.load(f)
    return [dict(r, image_path=Path(r["image_path"])) for r in records]

#### Section 1.3.7 - Individual Image Analysis and Metrics per Threshold

Across each threshold we are looping over, we need a way to filter whether our image's confidence exceeds the threshold and accept the image for use in our metrics, and so we use `filter_predictions` which takes our images and the current threshold and prunes images less than our threshold. This decreases the set of images we use, and hence our accepted prediction set must be smaller than our original prediction set at each threshold. Then, for a given confidence threshold, we analyse whether the image's confidence exceeds the threshold and whether the IoU exceeds 0.5 for a given box, which ultimately determines whether the box is considered "dog-positive", calculated in `match_record`. 

Next, we initialise the metrics we will use to evaluate the performance of each model. We use the Precision-Recall method, as well as the F1 score and specificity: $$P = \frac{TP}{TP + FP}$$ measures the proportion of true positive predictions out of all positive predictions, $$R = \frac{TP}{TP + FN}$$ measures the proportion of true positive instances that were labelled true instead of falsely negative, $$F_1 = \frac{2TP}{2TP + FP + FN}$$ is the harmonic mean of precision and recall, and $$S = \frac{TN}{TN + FP}$$ is the proportion of images without dogs that were correctly rejected (calculated later). These are all made in `calculate_metrics` given the number of each classification, and uses `safe_divide` so as to not divide by 0. 

To obtain the `tp, fp, fn, tn` values, we apply `evaluate_threshold` which returns all values of the confusion matrix for a given threshold, and does this for both image-level and box-level validation. Across all thresholds, we run `evaluate_thresholds` to create a combined set of information about how the model performs across different confidences. 

It makes sense that if we - the dog detectors - are passing cropped images of dogs to the emotion detectors, it is imperative that they receive only dogs, as classifying a different object's emotion. Therefore, we aim to minimise the number of false positives without compromising the number of true positives, and thus we select the threshold that yields the maximum box F1 score as a sufficient threshold for our subjective requirements: $$t^* = \arg\max_{t\in T}F_{1, \text{box}}(t)$$ if our threshold set is $T$ with elements $t \in (0.05, 0.95)$ with 0.05 gaps. We then save all threshold metrics in a CSV to be used for plot creation. 

In [ ]:
def safe_divide(a, b):
    return a / b if b else 0.0

def filter_predictions(predictions, threshold):
    return [p for p in predictions if p["confidence"] >= threshold]


def match_record(record, threshold, iou_threshold):
    active = filter_predictions(record["predictions"], threshold)
    matched_pred, matched_gt = match_boxes(active, record["gt_boxes"], iou_threshold)
    return dict(record, active_predictions=active, matched_pred=matched_pred,
                matched_gt=matched_gt, predicted_dog=record["max_confidence"] >= threshold)


def calculate_metrics(tp, fp, fn):
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    return precision, recall, safe_divide(2 * precision * recall, precision + recall)


def evaluate_threshold(records, threshold, iou_threshold):
    image_tp = image_fp = image_tn = image_fn = 0
    box_tp = box_fp = box_fn = 0
    for record in records:
        r = match_record(record, threshold, iou_threshold)
        actual, predicted = r["actual_dog"], r["predicted_dog"]
        if actual and predicted:
            image_tp += 1
        elif not actual and predicted:
            image_fp += 1
        elif not actual and not predicted:
            image_tn += 1
        else:
            image_fn += 1
        box_tp += len(r["matched_pred"])
        box_fp += len(r["active_predictions"]) - len(r["matched_pred"])
        box_fn += len(r["gt_boxes"]) - len(r["matched_gt"])
    image_precision, image_recall, image_f1 = calculate_metrics(image_tp, image_fp, image_fn)
    box_precision, box_recall, box_f1 = calculate_metrics(box_tp, box_fp, box_fn)
    return {"threshold": threshold, "image_TP": image_tp, "image_FP": image_fp,
            "image_TN": image_tn, "image_FN": image_fn, "image_precision": image_precision,
            "image_recall": image_recall, "image_specificity": safe_divide(image_tn, image_tn + image_fp),
            "image_f1": image_f1, "box_TP": box_tp, "box_FP": box_fp, "box_FN": box_fn,
            "box_precision": box_precision, "box_recall": box_recall, "box_f1": box_f1}


def evaluate_thresholds(records, thresholds, iou_threshold):
    return [evaluate_threshold(records, threshold, iou_threshold) for threshold in thresholds]


def select_optimal_threshold(threshold_results):
    # Same rule as previously agreed: highest box F1, then highest box precision.
    # If both tie, max returns the first row (ascending sweep: lowest threshold).
    return max(threshold_results, key=lambda row: (row["box_f1"], row["box_precision"]))


def save_threshold_metrics(threshold_results, output_dir):
    save_csv(output_dir / "threshold_metrics.csv", threshold_results)


def prepare_operating_records(records, threshold, iou_threshold):
    """Recompute matches once at the winner, then share them across all reports."""
    return [match_record(r, threshold, iou_threshold) for r in records]


#### Section 1.3.8 - Dog Cropping

In [ ]:
def save_image_results(records, output_dir):
    rows = []
    for r in records:
        actual, predicted = int(r["actual_dog"]), int(r["predicted_dog"])
        rows.append({"image": r["image_path"].name, "actual_dog": actual,
                     "predicted_dog": predicted, "max_dog_confidence": r["max_confidence"],
                     "number_of_detections": len(r["active_predictions"]),
                     "correct": actual == predicted})
    save_csv(output_dir / "image_results.csv", rows)


def crop_dogs_from_image(image_path, predictions, crop_dir):
    """Original unpadded crop and filename format; no second inference."""
    if not predictions:
        return
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Could not read {image_path}")
    h, w = image.shape[:2]
    for dog_number, pred in enumerate(predictions, start=1):
        x1, y1, x2, y2 = pred["box"]
        left = max(0, int(np.floor(x1)))
        top = max(0, int(np.floor(y1)))
        right = min(w, int(np.ceil(x2)))
        bottom = min(h, int(np.ceil(y2)))
        crop = image[top:bottom, left:right]
        if crop.size == 0:
            continue
        filename = f"{image_path.stem}_dog_{dog_number}_{pred['confidence']:.3f}.jpg"
        save_image(crop_dir / filename, crop)


def save_dog_crops(records, crop_dir):
    for r in records:
        crop_dogs_from_image(r["image_path"], r["active_predictions"], crop_dir)


#### Section 1.3.9 - Plotting and Confusion Matrices

In [ ]:
def plot_image_confusion_matrix(records, output_dir, model_name, operating_threshold):
    selected_actual = [int(r["actual_dog"]) for r in records]
    selected_predictions = [int(r["predicted_dog"]) for r in records]
    cm = confusion_matrix(
        selected_actual,
        selected_predictions,
        labels=[0, 1]
    )
    tn, fp, fn, tp = cm.ravel()
    print("\nIMAGE-LEVEL CONFUSION MATRIX")
    print("True positives:", tp)
    print("False positives:", fp)
    print("True negatives:", tn)
    print("False negatives:", fn)
    print(
        "Precision:",
        safe_divide(tp, tp + fp)
    )
    print(
        "Recall:",
        safe_divide(tp, tp + fn)
    )
    print(
        "Specificity:",
        safe_divide(tn, tn + fp)
    )
    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No dog", "Dog"]
    )
    display.plot(values_format="d")
    plt.title(
        f"{model_name}: Image-level Confusion Matrix\n"
        f"Confidence = {operating_threshold}"
    )
    plt.tight_layout()
    plt.savefig(
        output_dir / "confusion_matrix.png",
        dpi=200
    )
    plt.close()


def plot_box_confusion_matrix(selected_row, output_dir, model_name, iou_threshold):
    # Extract existing bounding-box counts
    tp = selected_row["box_TP"]
    fp = selected_row["box_FP"]
    fn = selected_row["box_FN"]
    # True negatives are undefined for object detection
    cm = np.array([
        [np.nan, fp],
        [fn, tp]
    ], dtype=float)
    # Create confusion matrix figure
    fig, ax = plt.subplots(figsize=(7, 6))
    cmap = plt.get_cmap("Blues").copy()
    cmap.set_bad(color="lightgrey")
    im = ax.imshow(
        np.ma.masked_invalid(cm),
        cmap=cmap,
        vmin=0,
        vmax=max(tp, fp, fn, 1)
    )
    # Set class labels
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["No dog", "Dog"])
    ax.set_yticklabels(["No dog", "Dog"])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(
        f"{model_name}: Box-level Confusion Matrix\n"
        f"Confidence = {selected_row['threshold']:.2f} | "
        f"IoU = {iou_threshold:.2f}"
    )
    # Annotate matrix cells
    labels = [
        ["N/A", str(fp)],
        [str(fn), str(tp)]
    ]
    max_value = max(tp, fp, fn, 1)
    for i in range(2):
        for j in range(2):
            value = cm[i, j]
            colour = (
                "white"
                if not np.isnan(value)
                and value > max_value / 2
                else "black"
            )
            ax.text(
                j,
                i,
                labels[i][j],
                ha="center",
                va="center",
                color=colour,
                fontsize=14
            )
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(
        output_dir / "box_confusion_matrix.png",
        dpi=200
    )
    plt.close(fig)
    # Print box-level results
    print("\nBOX-LEVEL CONFUSION MATRIX")
    print("True positives:", tp)
    print("False positives:", fp)
    print("False negatives:", fn)
    print("True negatives: N/A")
    print(
        "Box precision:",
        safe_divide(tp, tp + fp)
    )
    print(
        "Box recall:",
        safe_divide(tp, tp + fn)
    )
    print(
        "Box F1:",
        safe_divide(
            2 * tp,
            2 * tp + fp + fn
        )
    )


def plot_metrics_vs_confidence(threshold_results, output_dir, model_name,
                               operating_threshold, level):
    if level == "image":
        metrics = [("image_precision", "Precision"), ("image_recall", "Recall"),
                   ("image_specificity", "Specificity"), ("image_f1", "F1")]
        title = f"{model_name}: Image-level performance"
    elif level == "box":
        metrics = [("box_precision", "Box Precision"), ("box_recall", "Box Recall"),
                   ("box_f1", "Box F1")]
        title = f"{model_name}: Box-level performance"
    else:
        raise ValueError(f"Unknown metric level: {level}")
    thresholds = [r["threshold"] for r in threshold_results]
    plt.figure(figsize=(10, 6))
    for metric, label in metrics:
        values = [r[metric] for r in threshold_results]
        plt.plot(thresholds, values, marker="o", label=label)
    plt.axvline(operating_threshold, linestyle="--", label="Selected threshold")
    plt.xlabel("Confidence threshold")
    plt.ylabel("Score")
    plt.title(title)
    plt.ylim(0, 1.05)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(output_dir / f"{level}_metrics_vs_confidence.png", dpi=200)
    plt.close()


def plot_image_metrics_vs_confidence(rows, output_dir, model_name, threshold):
    plot_metrics_vs_confidence(rows, output_dir, model_name, threshold, "image")


def plot_box_metrics_vs_confidence(rows, output_dir, model_name, threshold):
    plot_metrics_vs_confidence(rows, output_dir, model_name, threshold, "box")


def plot_confidence_distribution(records, output_dir, model_name, operating_threshold):
    positive_confidences = [
        r["max_confidence"]
        for r in records
        if r["actual_dog"]
    ]
    negative_confidences = [
        r["max_confidence"]
        for r in records
        if not r["actual_dog"]
    ]
    plt.figure(figsize=(10, 6))
    plt.hist(
        positive_confidences,
        bins=20,
        alpha=0.6,
        label="Images containing dogs"
    )
    plt.hist(
        negative_confidences,
        bins=20,
        alpha=0.6,
        label="Images without dogs"
    )
    plt.axvline(
        operating_threshold,
        linestyle="--",
        label="Selected threshold"
    )
    plt.xlabel("Maximum dog confidence per image")
    plt.ylabel("Number of images")
    plt.title(f"{model_name}: Confidence distributions")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(
        output_dir / "confidence_distribution.png",
        dpi=200
    )
    plt.close()

#### Section 1.3.10 - Bounding Box Analysis

In [ ]:
def investigate_bounding_boxes(records, output_dir):
    detection_rows, missed_gt_rows = [], []
    for record in records:
        gt_boxes = record["gt_boxes"]
        for i, pred in enumerate(record["active_predictions"]):
            best_iou = max((calculate_iou(pred["box"], gt) for gt in gt_boxes), default=0.0)
            matched = i in record["matched_pred"]
            detection_rows.append({"image": record["image_path"].name,
                                   "confidence": pred["confidence"], "best_IoU": best_iou,
                                   "matched_at_IoU_0.5": matched,
                                   "classification": "Matched dog box" if matched else "Unmatched prediction"})
        for gt_index, gt in enumerate(gt_boxes):
            if gt_index not in record["matched_gt"]:
                missed_gt_rows.append({"image": record["image_path"].name, "ground_truth_box": gt})
    save_csv(output_dir / "box_iou_results.csv", detection_rows,
             ["image", "confidence", "best_IoU", "matched_at_IoU_0.5", "classification"])
    save_csv(output_dir / "missed_ground_truth.csv", missed_gt_rows, ["image", "ground_truth_box"])


def save_box_diagnostic_image(record, diagnostic_dir):
    active_predictions = record["active_predictions"]
    gt_boxes = record["gt_boxes"]
    matched_pred, matched_gt = record["matched_pred"], record["matched_gt"]
    if len(matched_pred) == len(active_predictions) and len(matched_gt) == len(gt_boxes):
        return
    image = cv2.imread(
        str(record["image_path"])
    )
    # Draw ground-truth boxes in GREEN
    for gt in gt_boxes:
        x1, y1, x2, y2 = map(int, gt)
        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )
        cv2.putText(
            image,
            "GROUND TRUTH",
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            (0, 255, 0),
            1
        )
    # Draw predicted boxes in RED if unmatched,
    # and BLUE if they match a ground-truth box.
    for i, pred in enumerate(active_predictions):
        x1, y1, x2, y2 = map(
            int, pred["box"]
        )
        colour = (
            (255, 0, 0)
            if i in matched_pred
            else (0, 0, 255)
        )
        label = (
            f"PRED {pred['confidence']:.2f}"
        )
        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            colour,
            2
        )
        cv2.putText(
            image,
            label,
            (x1, min(image.shape[0] - 5, y2 + 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            colour,
            1
        )
    diagnostic_path = (
        diagnostic_dir /
        record["image_path"].name
    )
    save_image(diagnostic_path, image)


def save_box_diagnostics(records, diagnostic_dir):
    for record in records:
        save_box_diagnostic_image(record, diagnostic_dir)

#### Section 1.3.11 - Orchestration

In [ ]:
def print_summary(selected_row, model_name, output_dir):
    print("\nMODEL COMPLETE:", model_name)
    print("Selected confidence:", selected_row["threshold"])
    print("Image precision:", selected_row["image_precision"])
    print("Image recall:", selected_row["image_recall"])
    print("Image specificity:", selected_row["image_specificity"])
    print("Box precision:", selected_row["box_precision"])
    print("Box recall:", selected_row["box_recall"])
    print("Box F1:", selected_row["box_f1"])
    print("Output folder:", output_dir)


def generate_validation_outputs(records, output_dir, model_name):
    """Model-independent evaluation and report generation from cached predictions."""
    # The reference CSV has a fixed IoU-0.5 column name. Preserve the schema.
    if IOU_MATCH_THRESHOLD != 0.50:
        raise ValueError("Keep IOU_MATCH_THRESHOLD=0.50 to preserve the original CSV schema")
    threshold_results = evaluate_thresholds(records, THRESHOLDS, IOU_MATCH_THRESHOLD)
    save_threshold_metrics(threshold_results, output_dir)
    selected_row = select_optimal_threshold(threshold_results)
    operating_threshold = selected_row["threshold"]
    if selected_row["box_TP"] == 0:
        print("WARNING: No matched dog detections at the selected threshold. "
              "The F1 optimum is not a useful deployment operating point.")
    print(f"Best box-F1 threshold: {operating_threshold:.2f} | F1: {selected_row['box_f1']:.6f}")
    operating_records = prepare_operating_records(records, operating_threshold, IOU_MATCH_THRESHOLD)
    save_image_results(operating_records, output_dir)
    save_dog_crops(operating_records, output_dir / "dog_crops")
    plot_image_confusion_matrix(operating_records, output_dir, model_name, operating_threshold)
    plot_box_confusion_matrix(selected_row, output_dir, model_name, IOU_MATCH_THRESHOLD)
    plot_image_metrics_vs_confidence(threshold_results, output_dir, model_name, operating_threshold)
    plot_box_metrics_vs_confidence(threshold_results, output_dir, model_name, operating_threshold)
    plot_confidence_distribution(records, output_dir, model_name, operating_threshold)
    investigate_bounding_boxes(operating_records, output_dir)
    save_box_diagnostics(operating_records, output_dir / "box_diagnostics")
    # Future semantic false-positive investigation can be called here.
    print_summary(selected_row, model_name, output_dir)
    return threshold_results, selected_row


def evaluate_model(model_family, model_type, model_path, image_dir, label_dir,
                   dog_class_id_override=None):
    model_family, model_type = model_family.strip().upper(), model_type.strip().lower()
    image_paths = get_image_paths(Path(image_dir))
    print("Validation images:", len(image_paths))
    print(f"\nEVALUATING: {model_family} {model_type}")
    model = load_model(model_family, model_type, model_path)
    dog_class_id = get_dog_class_id(model, model_family, model_type, dog_class_id_override)
    print("Dog class ID:", dog_class_id if dog_class_id is not None else "checkpoint class-name metadata")
    output_dir = prepare_output_directory(model_family, model_type)
    records = collect_validation_records(model, image_paths, Path(label_dir), model_family, dog_class_id)
    save_prediction_records(records, output_dir)
    return generate_validation_outputs(records, output_dir, model_type)


def main():
    evaluate_model(MODEL_FAMILY, MODEL_TYPE, MODEL_PATH, VAL_IMAGE_DIR,
                   VAL_LABEL_DIR, DOG_CLASS_ID_OVERRIDE)
    print("\nALL MODEL EVALUATIONS COMPLETE!")

if __name__ == "__main__":
    main()

## Section 2 - Dog Emotion Classifier

## Section 3 - Combining Both Models

### Section 3.1 - Testing Data Creation

### Section 3.2 - Dog Detection and Cropping in Testing